# Training a Flux model

A small fully executable Flux training loop through `QuantumGraph.Trainer`.


In [ ]:
function find_repo_root(start = pwd())
    dir = abspath(start)
    while !isfile(joinpath(dir, "Project.toml")) || !isdir(joinpath(dir, "src"))
        parent = dirname(dir)
        parent == dir && error("Could not find QuantumGraph.jl repository root")
        dir = parent
    end
    dir
end

repo_root = find_repo_root()
cd(repo_root)
using Pkg
Pkg.activate(repo_root)


In [ ]:
using QuantumGraph
using DataFrames
using Flux
using Optimisers

x = Float32[1 2 3 4; 2 3 4 5]
y = Float32[3 5 7 9]
batches = [(x, y)]
model = Flux.Chain(Flux.Dense(2 => 8, Flux.relu), Flux.Dense(8 => 1))
loss(model, batch) = sum(abs2, vec(model(batch[1])) .- batch[2])
evaluator(model, iterator) = begin
    value = loss(model, first(iterator))
    DataFrame(loss_avg = [value], loss_min = [value], loss_max = [value])
end


In [ ]:
trainer = construct_trainer(Dict{String, Any}(
    "dataset" => batches,
    "model" => model,
    "optimizer" => Optimisers.Adam(0.01),
    "loss" => loss,
    "scheduler" => nothing,
    "evaluator" => evaluator,
    "early_stopping" => early_stopping_state(metric = :loss_avg),
    "output_path" => mktempdir(),
    "device" => "cpu",
    "num_epochs" => 3,
    "checkpoint_at" => 1,
))
fit_trainer!(trainer)
training_artifact_paths(trainer), last(trainer.reports)
